# FraudFinder - Feature Engineering with BigQuery (batch)

## Overview

This series of labs are updated upon [FraudFinder](https://github.com/googlecloudplatform/fraudfinder) repository which builds a end-to-end real-time fraud detection system on Google Cloud. Throughout the FraudFinder labs, you will learn how to read historical bank transaction data stored in data warehouse, read from a live stream of new transactions, perform exploratory data analysis (EDA), do feature engineering, ingest features into a feature store, train a model using feature store, register your model in a model registry, evaluate your model, deploy your model to an endpoint, do real-time inference on your model with feature store, and monitor your model.


In this notebook, we'll focus on a critical step in any machine learning project: **feature engineering**. You'll learn how to transform raw transaction data into meaningful features that can be used to train a powerful fraud detection model. We'll be using BigQuery for batch feature engineering and Vertex AI Feature Store to manage and serve our features.

### Objective

The primary goal of this notebook is to introduce you to the concepts of batch feature engineering and the Vertex AI Feature Store. You will learn how to:

* **Understand the difference between batch and streaming feature engineering.** We'll explore why both are essential for building a real-time fraud detection system.
* **Create powerful features using SQL in BigQuery.** You'll write queries to extract valuable insights from historical transaction data, such as customer spending habits and terminal risk profiles.

This lab uses the following Google Cloud services and resources:

- [Vertex AI](https://cloud.google.com/vertex-ai/)
- [BigQuery](https://cloud.google.com/bigquery/)

Steps performed in this notebook:

- Build customer and terminal-related features
- Organize features in a BigQuery table

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [1]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)


BUCKET_NAME          = "qwiklabs-asl-04-fb8d9ff3cdaf-fraudfinder"
PROJECT              = "qwiklabs-asl-04-fb8d9ff3cdaf"
REGION               = "us-central1"
ID                   = "ozrwj"
FEATURESTORE_ID      = "fraudfinder_ozrwj"
MODEL_NAME           = "ff_model"
ENDPOINT_NAME        = "ff_model_endpoint"
TRAINING_DS_SIZE     = "1000"



### Import libraries

In [2]:
# General
import datetime as dt
import json
import os
import random
import sys
import time
from datetime import datetime, timedelta
from typing import List, Union

# Data Engineering
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 500)

# Vertex AI and Vertex AI Feature Store
from google.cloud import aiplatform as vertex_ai
from google.cloud import bigquery

/home/jupyter/asl-ml-immersion/asl_mlops/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


### Define constants

In [3]:
# Define the date range of transactions for feature engineering (last 10 days up until yesterday)
YESTERDAY = datetime.today() - timedelta(days=1)
YEAR_MONTH_PREFIX = YESTERDAY.strftime("%Y-%m")
DATAPROCESSING_START_DATE = (YESTERDAY - timedelta(days=10)).strftime(
    "%Y-%m-%d"
)
DATAPROCESSING_END_DATE = YESTERDAY.strftime("%Y-%m-%d")

# Define BiqQuery dataset and tables to calculate features.
RAW_BQ_TRANSACTION_TABLE_URI = f"{PROJECT_ID}.tx.tx"

INGESTION_BQ_TRANSACTION_TABLE_URI = f"{PROJECT_ID}.tx.ingestion_tx_records"
INGESTION_BQ_LABELS_TABLE_URI = f"{PROJECT_ID}.tx.ingestion_tx_labels"

RAW_BQ_LABELS_TABLE_URI = f"{PROJECT_ID}.tx.txlabels"
FEATURES_BQ_TABLE_URI = f"{PROJECT_ID}.tx.wide_features_table"

# Define Vertex AI Feature store tables and views.

CUSTOMERS_FE_BQ_VIEW_URI = f"{PROJECT_ID}.tx.v_customers_features"
CUSTOMERS_FE_BQ_BATCH_TABLE_URI = f"{PROJECT_ID}.tx.t_customers_batch_features"

TERMINALS_TABLE_NAME = f"terminals_{DATAPROCESSING_END_DATE.replace('-', '')}"

TERMINALS_FE_BQ_VIEW_URI = f"{PROJECT_ID}.tx.v_terminals_features"
TERMINALS_FE_BQ_BATCH_TABLE_URI = f"{PROJECT_ID}.tx.t_terminals_batch_features"

CUSTOMERS_STREAMING_FE_TABLE_URI = (
    f"{PROJECT_ID}.tx.t_customers_streaming_features"
)
TERMINALS_STREAMING_FE_TABLE_URI = (
    f"{PROJECT_ID}.tx.t_terminals_streaming_features"
)

ONLINE_STORAGE_NODES = 1
FEATURE_TIME = "feature_ts"
CUSTOMER_ENTITY_ID = "customer"
TERMINAL_ENTITY_ID = "terminal"

### Helpers

Define a set of helper functions to run BigQuery query and create features. 

In [4]:
def run_bq_query(sql: str, show=False) -> Union[str, pd.DataFrame]:
    """
    Run a BigQuery query and return the job ID or result as a DataFrame
    Args:
        sql: SQL query, as a string, to execute in BigQuery
        show: A flag to show query result in a Pandas Dataframe
    Returns:
        df: DataFrame of results from query,  or error, if any
    """

    bq_client = bigquery.Client()

    # Try dry run before executing query to catch any errors
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    bq_client.query(sql, job_config=job_config)

    # If dry run succeeds without errors, proceed to run query
    job_config = bigquery.QueryJobConfig()
    client_result = bq_client.query(sql, job_config=job_config)

    job_id = client_result.job_id

    # Wait for query/job to finish running. then get & return data frame
    result = client_result.result()
    print(f"Finished job_id: {job_id}")

    if show:
        df = result.to_arrow().to_pandas()
        return df

## Creating Destination Tables for the Ingestion Pipeline

Before we start engineering features, we need a place to store our raw transaction data. In this section, we'll create two BigQuery tables to serve as the destination for our ingestion pipeline. These tables will store the raw transaction records and their corresponding labels (i.e., whether a transaction is fraudulent or not). This separation of raw data from engineered features is a good practice that helps with data organization and reusability.

### Creating a Table for Raw Transaction Records

In [5]:
create_ingestion_tx_records_table = f"""
CREATE OR REPLACE TABLE `{INGESTION_BQ_TRANSACTION_TABLE_URI}`
(
  TX_ID STRING OPTIONS(description="Unique transaction identifier"),
  TX_TS TIMESTAMP OPTIONS(description="Timestamp of the transaction"),
  CUSTOMER_ID STRING OPTIONS(description="Unique customer identifier"),
  TERMINAL_ID STRING OPTIONS(description="Unique terminal identifier"),
  TX_AMOUNT FLOAT64 OPTIONS(description="The monetary value of the transaction")
)
PARTITION BY
  DATE(TX_TS)
CLUSTER BY
  CUSTOMER_ID
OPTIONS (
  description = "A table to store customer transaction data, partitioned by day and clustered by customer."
)"""
print(create_ingestion_tx_records_table)


CREATE OR REPLACE TABLE `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records`
(
  TX_ID STRING OPTIONS(description="Unique transaction identifier"),
  TX_TS TIMESTAMP OPTIONS(description="Timestamp of the transaction"),
  CUSTOMER_ID STRING OPTIONS(description="Unique customer identifier"),
  TERMINAL_ID STRING OPTIONS(description="Unique terminal identifier"),
  TX_AMOUNT FLOAT64 OPTIONS(description="The monetary value of the transaction")
)
PARTITION BY
  DATE(TX_TS)
CLUSTER BY
  CUSTOMER_ID
OPTIONS (
  description = "A table to store customer transaction data, partitioned by day and clustered by customer."
)


In [6]:
run_bq_query(create_ingestion_tx_records_table)

Finished job_id: e98e1846-0566-476f-bbb9-3abc666423e4


### Creating Table for input labels records

In [7]:
create_ingestion_tx_labels_table = f"""
CREATE OR REPLACE TABLE `{INGESTION_BQ_LABELS_TABLE_URI}`
(
  TX_ID STRING OPTIONS(description="Unique transaction identifier"),
  TX_FRAUD INT64 OPTIONS(description="The label for the transaction, 1-fraud/ 0-not a fraud")
)
OPTIONS (
  description = "A table to store fraud labels for transaction data"
)"""
print(create_ingestion_tx_labels_table)


CREATE OR REPLACE TABLE `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_labels`
(
  TX_ID STRING OPTIONS(description="Unique transaction identifier"),
  TX_FRAUD INT64 OPTIONS(description="The label for the transaction, 1-fraud/ 0-not a fraud")
)
OPTIONS (
  description = "A table to store fraud labels for transaction data"
)


In [8]:
run_bq_query(create_ingestion_tx_labels_table)

Finished job_id: 6d3636dc-6bc3-47b7-b0e6-4914e9f6120e


## Feature Engineering

### Batch Feature Engineering with BigQuery

Now it's time to dive into the core of this notebook: **batch feature engineering**. In this section, we'll use the power of SQL and BigQuery to create insightful features from our historical transaction data. These features will capture patterns in customer behavior and terminal activity that can help our machine learning model distinguish between legitimate and fraudulent transactions.

We'll be creating two main types of features:

**1. Customer-Related Features:** These features will focus on the spending habits of individual customers. By analyzing their transaction history over different time windows (e.g., the last 1, 7, and 14 days), we can identify unusual patterns that might indicate fraud. For example, a sudden spike in the number of transactions or the average transaction amount could be a red flag.

**2. Terminal-Related Features:** These features will assess the risk associated with different transaction terminals. Some terminals might be more susceptible to fraud than others. By analyzing the history of fraudulent transactions at each terminal, we can create a risk score that can be used as a powerful feature in our model.

To create these features, we'll be using SQL window functions in BigQuery. These functions allow us to perform calculations across a set of table rows that are somehow related to the current row. This is perfect for our use case, as it allows us to easily calculate aggregated statistics (e.g., the average transaction amount) over different time windows.

Let's get started! In the following cells, we'll walk you through the process of creating these features step by step.

#### Creating Batch Features with SQL

Now, let's get our hands dirty and write some SQL! In the following cells, we'll define the queries to create our customer and terminal-related features. We'll be using Common Table Expressions (CTEs) to make our queries more readable and organized.

**Set the Date Range for Feature Engineering**

First, let's define the time window for which we want to create features. We'll use the last 10 days of transaction data as our source.

In [9]:
print(
    f"""
DATAPROCESSING_START_DATE: {DATAPROCESSING_START_DATE}
DATAPROCESSING_END_DATE: {DATAPROCESSING_END_DATE}
"""
)


DATAPROCESSING_START_DATE: 2026-05-10
DATAPROCESSING_END_DATE: 2026-05-20



### Append historical records to the input tables:

#### For transactions table:

In [10]:
insert_transactions_historical_data = f"""INSERT INTO `{INGESTION_BQ_TRANSACTION_TABLE_URI}`
 (TX_ID,
  TX_TS,
  CUSTOMER_ID,
  TERMINAL_ID,
  TX_AMOUNT)
SELECT
  TX_ID,
  TX_TS,
  CUSTOMER_ID,
  TERMINAL_ID,
  TX_AMOUNT
FROM `{RAW_BQ_TRANSACTION_TABLE_URI}`
WHERE TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()
"""
print(insert_transactions_historical_data)
run_bq_query(insert_transactions_historical_data)

INSERT INTO `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records`
 (TX_ID,
  TX_TS,
  CUSTOMER_ID,
  TERMINAL_ID,
  TX_AMOUNT)
SELECT
  TX_ID,
  TX_TS,
  CUSTOMER_ID,
  TERMINAL_ID,
  TX_AMOUNT
FROM `qwiklabs-asl-04-fb8d9ff3cdaf.tx.tx`
WHERE TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()

Finished job_id: d50492f5-7e08-419b-b15d-f45f5aba1c4e


for labels table:

#### For labels table:

In [11]:
insert_labels_historical_data = f"""INSERT INTO `{INGESTION_BQ_LABELS_TABLE_URI}`
 (TX_ID,
  TX_FRAUD)
  SELECT
    raw_tx.TX_ID,
    raw_lb.TX_FRAUD
  FROM
      `{INGESTION_BQ_TRANSACTION_TABLE_URI}` as raw_tx
  INNER JOIN 
    `{RAW_BQ_LABELS_TABLE_URI}` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID
"""
print(insert_labels_historical_data)
run_bq_query(insert_labels_historical_data)

INSERT INTO `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_labels`
 (TX_ID,
  TX_FRAUD)
  SELECT
    raw_tx.TX_ID,
    raw_lb.TX_FRAUD
  FROM
      `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records` as raw_tx
  INNER JOIN 
    `qwiklabs-asl-04-fb8d9ff3cdaf.tx.txlabels` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID

Finished job_id: e62b1abb-7bf0-40dc-9946-b6b133496e96


### Create Batch Table

#### Customers feature table

Customer table SQL query string:

In [12]:
create_customer_batch_features_query = f"""
CREATE OR REPLACE TABLE `{CUSTOMERS_FE_BQ_BATCH_TABLE_URI}` AS
WITH
  -- CTE 1: Select raw transaction data from the source table
  get_raw_table AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT
  FROM `{INGESTION_BQ_TRANSACTION_TABLE_URI}`),

  -- CTE 2: Calculate customer spending behavior using window functions
  get_customer_spending_behaviour AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT,
    
    -- Calculate the number of transactions for each customer over 1, 7, and 14-day windows
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 86400 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_1DAY_WINDOW,
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_7DAY_WINDOW,
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_14DAY_WINDOW,
      
    -- Calculate the average transaction amount for each customer over 1, 7, and 14-day windows
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 86400 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_14DAY_WINDOW
  FROM get_raw_table)

-- Final SELECT statement: Create the customer features table
SELECT
  PARSE_TIMESTAMP("%Y-%m-%d %H:%M:%S", FORMAT_TIMESTAMP("%Y-%m-%d %H:%M:%S", TX_TS, "UTC")) as feature_timestamp,
  CUSTOMER_ID AS entity_id,
  CAST(CUSTOMER_ID_NB_TX_1DAY_WINDOW AS INT64) AS customer_id_nb_tx_1day_window,
  CAST(CUSTOMER_ID_NB_TX_7DAY_WINDOW AS INT64) AS customer_id_nb_tx_7day_window,
  CAST(CUSTOMER_ID_NB_TX_14DAY_WINDOW AS INT64) AS customer_id_nb_tx_14day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_1day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_7day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_14DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_14day_window
FROM
  get_customer_spending_behaviour
"""
print(create_customer_batch_features_query)


CREATE OR REPLACE TABLE `qwiklabs-asl-04-fb8d9ff3cdaf.tx.t_customers_batch_features` AS
WITH
  -- CTE 1: Select raw transaction data from the source table
  get_raw_table AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT
  FROM `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records`),

  -- CTE 2: Calculate customer spending behavior using window functions
  get_customer_spending_behaviour AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT,

    -- Calculate the number of transactions for each customer over 1, 7, and 14-day windows
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 86400 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_1DAY_WINDOW,
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_7DAY_WINDOW,
    COUNT(TX_ID) OVER (PARTITION BY CUSTOMER_ID ORDER BY

##### Run the query 

You create the initial customer features table based on provided historical records snapshot

In [13]:
run_bq_query(create_customer_batch_features_query)

Finished job_id: 3d0211d2-31b7-435b-9c29-faebaa9594f2


##### Inspect the result 

You can query some data rows to validate the result of the query

In [14]:
run_bq_query(
    f"SELECT * FROM `{CUSTOMERS_FE_BQ_BATCH_TABLE_URI}` LIMIT 10", show=True
)

Finished job_id: 90464680-9277-429f-8fbf-e47af4ce2a3e


,feature_timestamp,entity_id,customer_id_nb_tx_1day_window,customer_id_nb_tx_7day_window,customer_id_nb_tx_14day_window,customer_id_avg_amount_1day_window,customer_id_avg_amount_7day_window,customer_id_avg_amount_14day_window
0,2026-05-08 05:12:58+00:00,0032042347915706,2,3,3,55.735000,54.713333,54.713333
1,2026-05-08 20:55:30+00:00,0032042347915706,3,5,5,46.750000,51.302000,51.302000
2,2026-05-15 05:55:49+00:00,0189956639198831,3,18,25,66.413333,65.435000,64.657200
3,2026-05-13 12:44:25+00:00,0195121366595185,5,13,14,14.674000,14.108462,14.097143
4,2026-05-14 00:48:31+00:00,0197074514291717,3,7,38,134.013333,115.680000,166.652895
5,2026-05-18 05:02:31+00:00,0263534426277810,2,16,32,26.555000,30.080000,30.573750
6,2026-05-19 21:42:23+00:00,0263534426277810,2,19,38,26.715000,28.880000,30.156842
7,2026-05-10 21:02:19+00:00,0291762451225654,6,16,16,6.090000,7.740000,7.740000
8,2026-05-08 18:28:17+00:00,0356215679795497,2,3,3,91.815000,92.053333,92.053333
9,2026-05-08 22:07:07+00:00,0377432776427351,2,3,3,80.040000,79.983333,79.983333


#### Terminal feature table

Terminal table SQL query string:

In [15]:
create_terminal_batch_features_query = f"""
CREATE OR REPLACE TABLE `{TERMINALS_FE_BQ_BATCH_TABLE_URI}` AS
WITH
  -- CTE 1: Join transaction data with fraud labels
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT,
    raw_lb.TX_FRAUD
  FROM `{INGESTION_BQ_TRANSACTION_TABLE_URI}` raw_tx
  LEFT JOIN 
    `{INGESTION_BQ_LABELS_TABLE_URI}` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID),

  -- CTE 2: Calculate delayed window variables for terminal risk assessment
  get_variables_delay_window AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    
    -- Calculate the number of fraudulent transactions and total transactions over a 7-day delay period
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_DELAY,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS NB_TX_DELAY,
      
    -- Calculate the number of fraudulent transactions and total transactions over 1, 7, and 14-day delayed windows
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 691200 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_1_DELAY_WINDOW,
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_7_DELAY_WINDOW,
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1814400 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_14_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 691200 PRECEDING
      AND CURRENT ROW ) AS NB_TX_1_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS NB_TX_7_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1814400 PRECEDING
      AND CURRENT ROW ) AS NB_TX_14_DELAY_WINDOW
  FROM get_raw_table),

  -- CTE 3: Calculate terminal risk factors
  get_risk_factors AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    -- Calculate the number of fraudulent transactions for each terminal over 1, 7, and 14-day windows
    NB_FRAUD_1_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_1DAY_WINDOW,
    NB_FRAUD_7_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_7DAY_WINDOW,
    NB_FRAUD_14_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_14DAY_WINDOW,
    -- Calculate the total number of transactions for each terminal over 1, 7, and 14-day windows
    NB_TX_1_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_1DAY_WINDOW,
    NB_TX_7_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_7DAY_WINDOW,
    NB_TX_14_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_14DAY_WINDOW
      FROM
    get_variables_delay_window),

  -- CTE 4: Calculate the terminal risk index
  get_risk_index AS (
    SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TERMINAL_ID_NB_TX_1DAY_WINDOW,
    TERMINAL_ID_NB_TX_7DAY_WINDOW,
    TERMINAL_ID_NB_TX_14DAY_WINDOW,
    -- Calculate the risk index for each terminal over 1, 7, and 14-day windows
    (TERMINAL_ID_NB_FRAUD_1DAY_WINDOW/(TERMINAL_ID_NB_TX_1DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_1DAY_WINDOW,
    (TERMINAL_ID_NB_FRAUD_7DAY_WINDOW/(TERMINAL_ID_NB_TX_7DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_7DAY_WINDOW,
    (TERMINAL_ID_NB_FRAUD_14DAY_WINDOW/(TERMINAL_ID_NB_TX_14DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_14DAY_WINDOW
    FROM get_risk_factors 
  )

-- Final SELECT statement: Create the terminal features table
SELECT
  PARSE_TIMESTAMP("%Y-%m-%d %H:%M:%S", FORMAT_TIMESTAMP("%Y-%m-%d %H:%M:%S", TX_TS, "UTC")) as feature_timestamp,
  TERMINAL_ID AS entity_id,
  CAST(TERMINAL_ID_NB_TX_1DAY_WINDOW AS INT64) AS terminal_id_nb_tx_1day_window,
  CAST(TERMINAL_ID_NB_TX_7DAY_WINDOW AS INT64) AS terminal_id_nb_tx_7day_window,
  CAST(TERMINAL_ID_NB_TX_14DAY_WINDOW AS INT64) AS terminal_id_nb_tx_14day_window,
  CAST(TERMINAL_ID_RISK_1DAY_WINDOW AS FLOAT64) AS terminal_id_risk_1day_window,
  CAST(TERMINAL_ID_RISK_7DAY_WINDOW AS FLOAT64) AS terminal_id_risk_7day_window,
  CAST(TERMINAL_ID_RISK_14DAY_WINDOW AS FLOAT64) AS terminal_id_risk_14day_window
FROM
  get_risk_index
"""
print(create_terminal_batch_features_query)


CREATE OR REPLACE TABLE `qwiklabs-asl-04-fb8d9ff3cdaf.tx.t_terminals_batch_features` AS
WITH
  -- CTE 1: Join transaction data with fraud labels
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT,
    raw_lb.TX_FRAUD
  FROM `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records` raw_tx
  LEFT JOIN 
    `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_labels` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID),

  -- CTE 2: Calculate delayed window variables for terminal risk assessment
  get_variables_delay_window AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,

    -- Calculate the number of fraudulent transactions and total transactions over a 7-day delay period
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_DELAY,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_

##### Run the query 

You create the customer features table

In [16]:
run_bq_query(create_terminal_batch_features_query)

Finished job_id: 6ea8e1cc-2e2e-4f0c-992c-bedabf7835a7


##### Inspect the result 

You can query some data rows to validate the result of the query

In [17]:
run_bq_query(
    f"SELECT * FROM `{TERMINALS_FE_BQ_BATCH_TABLE_URI}` LIMIT 10", show=True
)

Finished job_id: af1b6fdc-deae-4c7c-bd23-025a57a955c2


,feature_timestamp,entity_id,terminal_id_nb_tx_1day_window,terminal_id_nb_tx_7day_window,terminal_id_nb_tx_14day_window,terminal_id_risk_1day_window,terminal_id_risk_7day_window,terminal_id_risk_14day_window
0,2026-05-09 03:44:23+00:00,01014138,0,0,0,0.000000,0.000000,0.000000
1,2026-05-15 01:04:51+00:00,01014138,32,51,51,0.000000,0.000000,0.000000
2,2026-05-11 08:55:55+00:00,01109961,0,0,0,0.000000,0.000000,0.000000
3,2026-05-15 01:15:52+00:00,01109961,20,35,35,0.000000,0.000000,0.000000
4,2026-05-15 07:26:11+00:00,01109961,22,39,39,0.000000,0.000000,0.000000
5,2026-05-17 12:02:47+00:00,01109961,26,93,93,0.000000,0.010753,0.010753
6,2026-05-19 12:29:18+00:00,01109961,20,136,136,0.100000,0.022059,0.022059
7,2026-05-17 16:33:48+00:00,01485517,15,76,76,0.000000,0.000000,0.000000
8,2026-05-18 07:50:58+00:00,01765860,24,102,102,0.166666,0.039216,0.039216
9,2026-05-09 23:40:00+00:00,03123316,0,0,0,0.000000,0.000000,0.000000


### Creating BigQuery Views for Feature Tables

Now that we have created the initial batch feature tables, we will create BigQuery views on top of them. A **BigQuery view** is a virtual table defined by a SQL query. It allows you to encapsulate the logic for generating features and provides a simplified, consistent interface for querying them. Views do not store any data themselves; instead, they run the underlying query every time they are accessed, ensuring that you always get the latest data.

In this notebook, we will create two views:
- `v_customers_features`: A view for the customer-related batch features.
- `v_terminals_features`: A view for the terminal-related batch features.

These views will be used as the source for our Vertex AI Feature Store, and they will also be used to create our final training dataset.

#### Customer feature 

This view will provide a real-time look at customer spending behavior.

In [18]:
create_customer_view_query = f"""
CREATE OR REPLACE VIEW `{CUSTOMERS_FE_BQ_VIEW_URI}` AS
WITH
  -- query to join labels with features -------------------------------------------------------------------------------------------
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT
  FROM (
    SELECT
      *
    FROM
      `{INGESTION_BQ_TRANSACTION_TABLE_URI}`
    WHERE
      TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()
    ) raw_tx),

  -- query to calculate CUSTOMER spending behaviour --------------------------------------------------------------------------------
  get_customer_spending_behaviour AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT,
    
    # calc the number of customer tx over daily windows per customer (1, 7 and 15 days, expressed in seconds)
    COUNT(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 86400 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_1DAY_WINDOW,
    COUNT(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_7DAY_WINDOW,
    COUNT(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_NB_TX_14DAY_WINDOW,
      
    # calc the customer average tx amount over daily windows per customer (1, 7 and 15 days, expressed in seconds, in dollars ($))
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 86400 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,
    AVG(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS CUSTOMER_ID_AVG_AMOUNT_14DAY_WINDOW,
  FROM get_raw_table)

# Create the table with CUSTOMER  features ----------------------------------------------------------------------------
SELECT
  current_timestamp() as feature_timestamp,
  CUSTOMER_ID AS entity_id,
  CAST(CUSTOMER_ID_NB_TX_1DAY_WINDOW AS INT64) AS customer_id_nb_tx_1day_window,
  CAST(CUSTOMER_ID_NB_TX_7DAY_WINDOW AS INT64) AS customer_id_nb_tx_7day_window,
  CAST(CUSTOMER_ID_NB_TX_14DAY_WINDOW AS INT64) AS customer_id_nb_tx_14day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_1day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_7day_window,
  CAST(CUSTOMER_ID_AVG_AMOUNT_14DAY_WINDOW AS FLOAT64) AS customer_id_avg_amount_14day_window,
FROM
  get_customer_spending_behaviour
"""

In [19]:
print(create_customer_view_query)


CREATE OR REPLACE VIEW `qwiklabs-asl-04-fb8d9ff3cdaf.tx.v_customers_features` AS
WITH
  -- query to join labels with features -------------------------------------------------------------------------------------------
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT
  FROM (
    SELECT
      *
    FROM
      `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records`
    WHERE
      TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()
    ) raw_tx),

  -- query to calculate CUSTOMER spending behaviour --------------------------------------------------------------------------------
  get_customer_spending_behaviour AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TX_AMOUNT,

    # calc the number of customer tx over daily windows per customer (1, 7 and 15 days, expressed in seconds)
    COUNT(TX_AMOUNT) OVER (PARTITION BY CUSTOMER_ID ORDER BY UN

##### Run the query 

You create the customer features table

In [20]:
run_bq_query(create_customer_view_query)

Finished job_id: 55461889-c9c2-4dd5-916b-306b559a66d5


##### Inspect the result 

You can query some data rows to validate the result of the query

In [21]:
run_bq_query(f"SELECT * FROM `{CUSTOMERS_FE_BQ_VIEW_URI}` LIMIT 10", show=True)

Finished job_id: dd02aafc-c02a-4312-9bee-e2da90c067a0


,feature_timestamp,entity_id,customer_id_nb_tx_1day_window,customer_id_nb_tx_7day_window,customer_id_nb_tx_14day_window,customer_id_avg_amount_1day_window,customer_id_avg_amount_7day_window,customer_id_avg_amount_14day_window
0,2026-05-21 09:47:04.844234+00:00,0000149585117869,1,1,1,21.100000,21.100000,21.100000
1,2026-05-21 09:47:04.844234+00:00,0000149585117869,1,2,2,16.580000,18.840000,18.840000
2,2026-05-21 09:47:04.844234+00:00,0000875764023990,1,1,1,79.470000,79.470000,79.470000
3,2026-05-21 09:47:04.844234+00:00,0000875764023990,2,2,2,73.330000,73.330000,73.330000
4,2026-05-21 09:47:04.844234+00:00,0000875764023990,1,3,3,59.890000,68.850000,68.850000
5,2026-05-21 09:47:04.844234+00:00,0000875764023990,1,1,4,78.620000,78.620000,71.292500
6,2026-05-21 09:47:04.844234+00:00,0001071169708317,1,1,1,49.670000,49.670000,49.670000
7,2026-05-21 09:47:04.844234+00:00,0001071169708317,2,2,2,51.055000,51.055000,51.055000
8,2026-05-21 09:47:04.844234+00:00,0001071169708317,3,3,3,51.816667,51.816667,51.816667
9,2026-05-21 09:47:04.844234+00:00,0001071169708317,4,4,4,53.077500,53.077500,53.077500


#### Terminal feature table

Terminal table SQL query string:

In [22]:
create_terminal_view_query = f"""
# query to calculate TERMINAL spending behaviour --------------------------------------------------------------------------------
CREATE OR REPLACE VIEW `{TERMINALS_FE_BQ_VIEW_URI}` AS
WITH
  -- query to join labels with features -------------------------------------------------------------------------------------------
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT,
    raw_lb.TX_FRAUD
  FROM (
    SELECT
      *
    FROM
      `{INGESTION_BQ_TRANSACTION_TABLE_URI}`
    WHERE
      TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()
    ) raw_tx
  LEFT JOIN 
    `{INGESTION_BQ_LABELS_TABLE_URI}` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID),

  # query to calculate TERMINAL spending behaviour --------------------------------------------------------------------------------
  get_variables_delay_window AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    
    # calc total amount of fraudulent tx and the total number of tx over the delay period per terminal (7 days - delay, expressed in seconds)
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_DELAY,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 604800 PRECEDING
      AND CURRENT ROW ) AS NB_TX_DELAY,
      
    # calc total amount of fraudulent tx and the total number of tx over the delayed window per terminal (window + 7 days - delay, expressed in seconds)
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 691200 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_1_DELAY_WINDOW,
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_7_DELAY_WINDOW,
    SUM(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1814400 PRECEDING
      AND CURRENT ROW ) AS NB_FRAUD_14_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 691200 PRECEDING
      AND CURRENT ROW ) AS NB_TX_1_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1209600 PRECEDING
      AND CURRENT ROW ) AS NB_TX_7_DELAY_WINDOW,
    COUNT(TX_FRAUD) OVER (PARTITION BY TERMINAL_ID ORDER BY UNIX_SECONDS(TX_TS) ASC RANGE BETWEEN 1814400 PRECEDING
      AND CURRENT ROW ) AS NB_TX_14_DELAY_WINDOW,
  FROM get_raw_table),

  # query to calculate TERMINAL risk factors ---------------------------------------------------------------------------------------
  get_risk_factors AS (
  SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    # calculate numerator of risk index
    NB_FRAUD_1_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_1DAY_WINDOW,
    NB_FRAUD_7_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_7DAY_WINDOW,
    NB_FRAUD_14_DELAY_WINDOW - NB_FRAUD_DELAY AS TERMINAL_ID_NB_FRAUD_14DAY_WINDOW,
    # calculate denominator of risk index
    NB_TX_1_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_1DAY_WINDOW,
    NB_TX_7_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_7DAY_WINDOW,
    NB_TX_14_DELAY_WINDOW - NB_TX_DELAY AS TERMINAL_ID_NB_TX_14DAY_WINDOW,
      FROM
    get_variables_delay_window),

  # query to calculate the TERMINAL risk index -------------------------------------------------------------------------------------
  get_risk_index AS (
    SELECT
    TX_TS,
    TX_ID,
    CUSTOMER_ID,
    TERMINAL_ID,
    TERMINAL_ID_NB_TX_1DAY_WINDOW,
    TERMINAL_ID_NB_TX_7DAY_WINDOW,
    TERMINAL_ID_NB_TX_14DAY_WINDOW,
    # calculate the risk index
    (TERMINAL_ID_NB_FRAUD_1DAY_WINDOW/(TERMINAL_ID_NB_TX_1DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_1DAY_WINDOW,
    (TERMINAL_ID_NB_FRAUD_7DAY_WINDOW/(TERMINAL_ID_NB_TX_7DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_7DAY_WINDOW,
    (TERMINAL_ID_NB_FRAUD_14DAY_WINDOW/(TERMINAL_ID_NB_TX_14DAY_WINDOW+0.0001)) AS TERMINAL_ID_RISK_14DAY_WINDOW
    FROM get_risk_factors 
  )

# Create the table with CUSTOMER and TERMINAL features ----------------------------------------------------------------------------
SELECT
  current_timestamp() as feature_timestamp,
  # TERMINAL_ID AS terminal_id,
  TERMINAL_ID AS entity_id,
  CAST(TERMINAL_ID_NB_TX_1DAY_WINDOW AS INT64) AS terminal_id_nb_tx_1day_window,
  CAST(TERMINAL_ID_NB_TX_7DAY_WINDOW AS INT64) AS terminal_id_nb_tx_7day_window,
  CAST(TERMINAL_ID_NB_TX_14DAY_WINDOW AS INT64) AS terminal_id_nb_tx_14day_window,
  CAST(TERMINAL_ID_RISK_1DAY_WINDOW AS FLOAT64) AS terminal_id_risk_1day_window,
  CAST(TERMINAL_ID_RISK_7DAY_WINDOW AS FLOAT64) AS terminal_id_risk_7day_window,
  CAST(TERMINAL_ID_RISK_14DAY_WINDOW AS FLOAT64) AS terminal_id_risk_14day_window,
FROM
  get_risk_index
"""

In [23]:
print(create_terminal_view_query)


# query to calculate TERMINAL spending behaviour --------------------------------------------------------------------------------
CREATE OR REPLACE VIEW `qwiklabs-asl-04-fb8d9ff3cdaf.tx.v_terminals_features` AS
WITH
  -- query to join labels with features -------------------------------------------------------------------------------------------
  get_raw_table AS (
  SELECT
    raw_tx.TX_TS,
    raw_tx.TX_ID,
    raw_tx.CUSTOMER_ID,
    raw_tx.TERMINAL_ID,
    raw_tx.TX_AMOUNT,
    raw_lb.TX_FRAUD
  FROM (
    SELECT
      *
    FROM
      `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records`
    WHERE
      TX_TS BETWEEN TIMESTAMP_SUB(current_timestamp(), INTERVAL 15 DAY) AND current_timestamp()
    ) raw_tx
  LEFT JOIN 
    `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_labels` as raw_lb
  ON raw_tx.TX_ID = raw_lb.TX_ID),

  # query to calculate TERMINAL spending behaviour --------------------------------------------------------------------------------
  get_variables_delay_window 

##### Run the query 

You create the customer features table

In [24]:
run_bq_query(create_terminal_view_query)

Finished job_id: bbdb11b3-62ca-45a5-9ca5-00e380239a66


##### Inspect the result 

You can query some data rows to validate the result of the query

In [25]:
run_bq_query(f"SELECT * FROM `{TERMINALS_FE_BQ_VIEW_URI}` LIMIT 10", show=True)

Finished job_id: d0f3fa63-9e94-4a04-b620-65f99ce78767


,feature_timestamp,entity_id,terminal_id_nb_tx_1day_window,terminal_id_nb_tx_7day_window,terminal_id_nb_tx_14day_window,terminal_id_risk_1day_window,terminal_id_risk_7day_window,terminal_id_risk_14day_window
0,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
1,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
2,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
3,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
4,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
5,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
6,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
7,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
8,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0
9,2026-05-21 09:47:08.342764+00:00,00064542,0,0,0,0.0,0.0,0.0


### Automating Feature Updates with BigQuery Scheduled Queries

To ensure our batch features remain up-to-date, we need a way to periodically refresh them with the latest transaction data. Manually re-running our feature generation queries would be inefficient and error-prone. Instead, we can automate this process using **BigQuery scheduled queries**.

A scheduled query is a query that runs automatically on a recurring basis. In our case, we'll set up scheduled queries that run every 15 minutes. Each time they run, they will:

1.  Execute the query within our `v_customers_features` and `v_terminals_features` views to calculate the latest feature values.
2.  Append these new feature values to our batch feature tables (`t_customers_batch_features` and `t_terminals_batch_features`).

This ensures that our feature store always has access to fresh batch features, which is crucial for making accurate, timely fraud predictions. We'll use the `bq mk --transfer_config` command to create these scheduled queries.

In [26]:
!echo "{CUSTOMERS_FE_BQ_VIEW_URI}"

qwiklabs-asl-04-fb8d9ff3cdaf.tx.v_customers_features


#### Add Access for bigquerydatatransfer

In [27]:
PROJECT_NUMBER=!gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"

In [28]:
import re
PROJECT_NUMBER = re.sub(r'\D', '', str(PROJECT_NUMBER))

#### IMPORTANT: Open CloudShell terminal and execute the next command:

In [40]:
print(f"gcloud iam service-accounts add-iam-policy-binding {PROJECT_NUMBER}-compute@developer.gserviceaccount.com --member='serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-bigquerydatatransfer.iam.gserviceaccount.com' --role='roles/iam.serviceAccountTokenCreator'")

gcloud iam service-accounts add-iam-policy-binding 992044206639-compute@developer.gserviceaccount.com --member='serviceAccount:service-992044206639@gcp-sa-bigquerydatatransfer.iam.gserviceaccount.com' --role='roles/iam.serviceAccountTokenCreator'


This command need to configure permissions to Bigquery Datatransfer to schedule queries

#### Create scheduled queries to add fresh features:

In [41]:
query = f"INSERT INTO `{CUSTOMERS_FE_BQ_BATCH_TABLE_URI}` SELECT * FROM `{CUSTOMERS_FE_BQ_VIEW_URI}`;"
customer_params_str = json.dumps({"query": query})

!bq mk --transfer_config \
--data_source='scheduled_query' \
--display_name='Append Customers Features to Batch Features Table' \
--target_dataset='tx' \
--schedule='every 30 mins' \
--params='{customer_params_str}'

Transfer configuration 'projects/992044206639/locations/us-central1/transferConfigs/6a112c8f-0000-27da-afb7-582429a63f3c' successfully created.


In [42]:
query = f"INSERT INTO `{TERMINALS_FE_BQ_BATCH_TABLE_URI}` SELECT * FROM `{TERMINALS_FE_BQ_VIEW_URI}`;"
terminal_params_str = json.dumps({"query": query})

!bq mk --transfer_config \
--data_source='scheduled_query' \
--display_name='Append Terminal Features to Batch Features Table' \
--target_dataset='tx' \
--schedule='every 30 mins' \
--params='{terminal_params_str}'

Transfer configuration 'projects/992044206639/locations/us-central1/transferConfigs/6a10df0a-0000-2376-af7b-14223bc1fd6a' successfully created.


### Initializing Tables for Real-Time (Streaming) Features

While this notebook focuses on batch features, our end-to-end fraud detection system will also use real-time features calculated over very short time windows (e.g., the last 15, 30, and 60 minutes). These features will be generated by a streaming pipeline using Dataflow, which is covered in the next notebook (`04_feature_engineering_streaming.ipynb`).

However, we need to create the destination tables for these features *now*. The following queries will create empty tables in BigQuery with the correct schema for our future streaming features. This step is important because:

1.  **It defines the contract:** It establishes the schema that the streaming pipeline will write to.
2.  **It enables the training view:** It allows our final training dataset view (`v_ff_training_dataset`) to be created successfully, as it can reference these tables even though they are currently empty. The `IFNULL` function in the view will handle the absence of data, ensuring the query doesn't fail.

These tables will be populated with data once we run the Dataflow streaming pipeline in the next lab.

##### Customer feature table

Customer table SQL query string:

In [32]:
initiate_real_time_customer_features_query = f"""
CREATE OR REPLACE TABLE `{CUSTOMERS_STREAMING_FE_TABLE_URI}`
(
    entity_id STRING,
    feature_timestamp TIMESTAMP,
    customer_id_nb_tx_15min_window INT64,
    customer_id_nb_tx_30min_window INT64,
    customer_id_nb_tx_60min_window INT64,
    customer_id_avg_amount_15min_window FLOAT64,
    customer_id_avg_amount_30min_window FLOAT64,
    customer_id_avg_amount_60min_window FLOAT64
)
"""

In [33]:
initiate_real_time_terminal_features_query = f"""
CREATE OR REPLACE TABLE `{TERMINALS_STREAMING_FE_TABLE_URI}`
(
    entity_id STRING,
    feature_timestamp TIMESTAMP,
    terminal_id_nb_tx_15min_window INT64,
    terminal_id_nb_tx_30min_window INT64,
    terminal_id_nb_tx_60min_window INT64,
    terminal_id_avg_amount_15min_window FLOAT64,
    terminal_id_avg_amount_30min_window FLOAT64,
    terminal_id_avg_amount_60min_window FLOAT64
)
"""

#### Run the query above to initialize the real-time features.

In [34]:
for query in [
    initiate_real_time_customer_features_query,
    initiate_real_time_terminal_features_query,
]:
    run_bq_query(query)

Finished job_id: 93a476e1-7a70-4854-ad8d-a96135add756
Finished job_id: 4e917a2f-dd04-4001-aa3a-558dc67eb28b


#### Inspect BigQuery features tables

In [35]:
run_bq_query(
    f"SELECT * FROM `{CUSTOMERS_STREAMING_FE_TABLE_URI}` LIMIT 5", show=True
)

Finished job_id: 33c90600-6a33-43be-b3cd-9dd51effa179


,entity_id,feature_timestamp,customer_id_nb_tx_15min_window,customer_id_nb_tx_30min_window,customer_id_nb_tx_60min_window,customer_id_avg_amount_15min_window,customer_id_avg_amount_30min_window,customer_id_avg_amount_60min_window


In [36]:
run_bq_query(
    f"SELECT * FROM `{TERMINALS_STREAMING_FE_TABLE_URI}` LIMIT 5", show=True
)

Finished job_id: 3487fabd-bc5e-47ef-924d-6dd8710ca9d5


,entity_id,feature_timestamp,terminal_id_nb_tx_15min_window,terminal_id_nb_tx_30min_window,terminal_id_nb_tx_60min_window,terminal_id_avg_amount_15min_window,terminal_id_avg_amount_30min_window,terminal_id_avg_amount_60min_window


Let's look at the final schema of the features table:

### Creating the Final Training Dataset View

With our batch and streaming feature tables in place, we can now create a final BigQuery view that will serve as the source for our model training. This view, `v_ff_training_dataset`, will join the raw transaction data with the corresponding feature values from our batch and streaming tables.

A key challenge when creating a training dataset is ensuring that you are not introducing **data leakage**. Data leakage occurs when your training data contains information that would not be available at the time of prediction. For example, if we were to simply join our transaction data with the latest feature values, we would be leaking information from the future into the past.

To prevent this, we will use the `ML.ENTITY_FEATURES_AT_TIME` function in BigQuery. This function allows us to perform a **point-in-time lookup**, which means that for each transaction, we will retrieve the feature values that were valid at the time the transaction occurred. This ensures that our model is trained on the same data that it will see in a real-world prediction scenario, which is crucial for building a robust and accurate fraud detection model.

In [37]:
batch_customers_features_table = f"{PROJECT}.tx.t_customers_batch_features"
batch_terminals_features_table = f"{PROJECT}.tx.t_terminals_batch_features"

stream_customers_features_table = f"{PROJECT}.tx.t_customers_streaming_features"
stream_terminals_features_table = f"{PROJECT}.tx.t_terminals_streaming_features"

train_dataset_view_sql = f"""
CREATE OR REPLACE VIEW tx.v_ff_training_dataset AS
    WITH
      # -------------------------------------------
      # Using raw transaction table as a base table
      # filtered by specific time range
      # joined with labels data
      raw_tx_labled_range_table AS (
      SELECT
        raw_tx.TX_TS AS tx_timestamp,
        raw_tx.TX_ID AS transaction_id,   
        raw_tx.CUSTOMER_ID AS customer_id,
        raw_tx.TERMINAL_ID AS terminal_id,     
        raw_tx.TX_AMOUNT AS tx_amount,
        raw_lb.TX_FRAUD AS tx_fraud,
      FROM
        `{INGESTION_BQ_TRANSACTION_TABLE_URI}` AS raw_tx
      LEFT JOIN
        `{INGESTION_BQ_LABELS_TABLE_URI}` AS raw_lb
      ON
        raw_tx.TX_ID = raw_lb.TX_ID
      WHERE
        raw_tx.TX_TS BETWEEN TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY) AND CURRENT_TIMESTAMP()),
      # ---------------------------------------------
      # Using base transaction table
      # to create a entity_id with timestamp pairs
      # to lookup coresponding feautres for terminals
      terminals_time_table AS (
      SELECT
        raw_tx_labled_range_table.tx_timestamp AS `time`,
        raw_tx_labled_range_table.TERMINAL_ID AS entity_id,
      FROM
        raw_tx_labled_range_table),
      # ---------------------------------------------
      # Using base transaction table
      # to create a entity_id with timestamp pairs
      # to lookup coresponding feautres for customers
      customers_time_table AS (
      SELECT
        raw_tx_labled_range_table.tx_timestamp AS `time`,
        raw_tx_labled_range_table.CUSTOMER_ID AS entity_id,
      FROM
        raw_tx_labled_range_table)

    SELECT

    # Transaction timestamp with IDs:
    raw_tx_labled_range_table.tx_timestamp AS `timestamp`,
    raw_tx_labled_range_table.transaction_id,
    raw_tx_labled_range_table.customer_id,
    raw_tx_labled_range_table.terminal_id,

    # Lable for transaction:
    raw_tx_labled_range_table.tx_fraud,
    
    # Features from raw transaction:
    raw_tx_labled_range_table.tx_amount,

    # Features from customers batch pipeline:
    IFNULL(f_customers.customer_id_avg_amount_1day_window, 0.0) as customer_id_avg_amount_1day_window,
    IFNULL(f_customers.customer_id_avg_amount_7day_window, 0.0) as customer_id_avg_amount_7day_window,
    IFNULL(f_customers.customer_id_avg_amount_14day_window, 0.0) as customer_id_avg_amount_14day_window,
    IFNULL(CAST(f_customers.customer_id_nb_tx_1day_window AS FLOAT64), 0.0) as customer_id_nb_tx_1day_window,
    IFNULL(CAST(f_customers.customer_id_nb_tx_7day_window AS FLOAT64), 0.0) as customer_id_nb_tx_7day_window,
    IFNULL(CAST(f_customers.customer_id_nb_tx_14day_window AS FLOAT64), 0.0) as customer_id_nb_tx_14day_window,

    # Features from terminals batch pipeline:
    IFNULL(f_terminals.terminal_id_risk_1day_window, 0.0)  AS terminal_id_risk_1day_window,
    IFNULL(f_terminals.terminal_id_risk_7day_window, 0.0)  AS terminal_id_risk_7day_window,
    IFNULL(f_terminals.terminal_id_risk_14day_window, 0.0)  AS terminal_id_risk_14day_window,
    IFNULL(CAST(f_terminals.terminal_id_nb_tx_1day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_1day_window,
    IFNULL(CAST(f_terminals.terminal_id_nb_tx_7day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_7day_window,
    IFNULL(CAST(f_terminals.terminal_id_nb_tx_14day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_14day_window,
  
    # Features from customers streaming pipeline:
    IFNULL(f_customers_stream.customer_id_avg_amount_15min_window, 0.0) as customer_id_avg_amount_15min_window,
    IFNULL(f_customers_stream.customer_id_avg_amount_30min_window, 0.0) as customer_id_avg_amount_30min_window,
    IFNULL(f_customers_stream.customer_id_avg_amount_60min_window, 0.0) as customer_id_avg_amount_60min_window,
    IFNULL(CAST(f_customers_stream.customer_id_nb_tx_15min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_15min_window,
    IFNULL(CAST(f_customers_stream.customer_id_nb_tx_30min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_30min_window,
    IFNULL(CAST(f_customers_stream.customer_id_nb_tx_60min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_60min_window,
  
    # Features from terminals streaming pipeline:
    IFNULL(f_terminals_stream.terminal_id_avg_amount_15min_window, 0.0) as terminal_id_avg_amount_15min_window,
    IFNULL(f_terminals_stream.terminal_id_avg_amount_30min_window, 0.0) as terminal_id_avg_amount_30min_window,
    IFNULL(f_terminals_stream.terminal_id_avg_amount_60min_window, 0.0) as terminal_id_avg_amount_60min_window,
    IFNULL(CAST(terminal_id_nb_tx_15min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_15min_window,
    IFNULL(CAST(terminal_id_nb_tx_30min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_30min_window,
    IFNULL(CAST(f_terminals_stream.terminal_id_nb_tx_60min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_60min_window
      
    FROM
      raw_tx_labled_range_table
      
    LEFT JOIN
      ML.ENTITY_FEATURES_AT_TIME( TABLE `{batch_customers_features_table}`,
        TABLE customers_time_table,
        num_rows => 1,
        ignore_feature_nulls => TRUE) AS f_customers
    ON
      raw_tx_labled_range_table.customer_id = f_customers.entity_id
      AND raw_tx_labled_range_table.tx_timestamp = f_customers.feature_timestamp
      
    LEFT JOIN

      ML.ENTITY_FEATURES_AT_TIME( TABLE `{batch_terminals_features_table}`,
        TABLE terminals_time_table,
        num_rows => 1,
        ignore_feature_nulls => TRUE) AS f_terminals
    ON
      raw_tx_labled_range_table.terminal_id = f_terminals.entity_id
      AND raw_tx_labled_range_table.tx_timestamp = f_terminals.feature_timestamp
      
    LEFT JOIN
      ML.ENTITY_FEATURES_AT_TIME( TABLE `{stream_customers_features_table}`,
        TABLE customers_time_table,
        num_rows => 1,
        ignore_feature_nulls => TRUE) AS f_customers_stream
    ON
      raw_tx_labled_range_table.customer_id = f_customers_stream.entity_id
      AND raw_tx_labled_range_table.tx_timestamp = f_customers_stream.feature_timestamp
    LEFT JOIN
      ML.ENTITY_FEATURES_AT_TIME( TABLE `{stream_terminals_features_table}`,
        TABLE terminals_time_table,
        num_rows => 1,
        ignore_feature_nulls => TRUE) AS f_terminals_stream
    ON
      raw_tx_labled_range_table.terminal_id = f_terminals_stream.entity_id
      AND raw_tx_labled_range_table.tx_timestamp = f_terminals_stream.feature_timestamp
"""
print(train_dataset_view_sql)


CREATE OR REPLACE VIEW tx.v_ff_training_dataset AS
    WITH
      # -------------------------------------------
      # Using raw transaction table as a base table
      # filtered by specific time range
      # joined with labels data
      raw_tx_labled_range_table AS (
      SELECT
        raw_tx.TX_TS AS tx_timestamp,
        raw_tx.CUSTOMER_ID AS customer_id,
        raw_tx.TERMINAL_ID AS terminal_id,
        raw_tx.TX_AMOUNT AS tx_amount,
        raw_lb.TX_FRAUD AS tx_fraud,
      FROM
        `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_records` AS raw_tx
      LEFT JOIN
        `qwiklabs-asl-04-fb8d9ff3cdaf.tx.ingestion_tx_labels` AS raw_lb
      ON
        raw_tx.TX_ID = raw_lb.TX_ID
      WHERE
        raw_tx.TX_TS BETWEEN TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY) AND CURRENT_TIMESTAMP()),
      # ---------------------------------------------
      # Using base transaction table
      # to create a entity_id with timestamp pairs
      # to lookup coresponding feaut

In [38]:
run_bq_query(train_dataset_view_sql, show=True)

Finished job_id: 63397cb5-2c84-4412-91ed-11a8f8d7a2e1


""


#### Inspect BigQuery training dataset view:

In [39]:
run_bq_query("SELECT * FROM tx.v_ff_training_dataset LIMIT 5", show=True)

Finished job_id: e980557d-45ea-4900-a90e-c3081adc1695


,tx_amount,tx_fraud,timestamp,customer_id_avg_amount_1day_window,customer_id_avg_amount_7day_window,customer_id_avg_amount_14day_window,customer_id_nb_tx_1day_window,customer_id_nb_tx_7day_window,customer_id_nb_tx_14day_window,terminal_id_risk_1day_window,terminal_id_risk_7day_window,terminal_id_risk_14day_window,terminal_id_nb_tx_1day_window,terminal_id_nb_tx_7day_window,terminal_id_nb_tx_14day_window,customer_id_avg_amount_15min_window,customer_id_avg_amount_30min_window,customer_id_avg_amount_60min_window,customer_id_nb_tx_15min_window,customer_id_nb_tx_30min_window,customer_id_nb_tx_60min_window,terminal_id_avg_amount_15min_window,terminal_id_avg_amount_30min_window,terminal_id_avg_amount_60min_window,terminal_id_nb_tx_15min_window,terminal_id_nb_tx_30min_window,terminal_id_nb_tx_60min_window
0,15.52,0,2026-05-21 03:10:26+00:00,16.330,16.100303,15.470962,6.0,33.0,52.0,0.04,0.029586,0.038674,25.0,169.0,181.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,53.13,0,2026-05-21 07:19:47+00:00,53.130,46.810000,47.450323,1.0,13.0,31.0,0.00,0.015625,0.014218,18.0,192.0,211.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,67.88,0,2026-05-21 03:46:48+00:00,67.880,73.820000,74.011429,1.0,6.0,7.0,0.00,0.007576,0.006897,16.0,132.0,145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,42.14,0,2026-05-21 03:33:20+00:00,48.922,51.373182,51.636327,5.0,22.0,49.0,0.00,0.026738,0.023810,31.0,187.0,210.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,47.73,0,2026-05-21 07:14:34+00:00,46.998,51.214783,51.483061,5.0,23.0,49.0,0.00,0.015873,0.014218,25.0,189.0,211.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### END

Now you can go to the next notebook `03_feature_serving_featurestore.ipynb`

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.